In [53]:
import onnxruntime as ort
import numpy as np
import cv2

In [54]:
#Load model
MODEL_PATH = "../yolo_models/hand-trained.onnx"
IMAGE_PATH = "../images/many_people_raising_hands.jpg"

In [55]:
#Load the hand image
image = cv2.imread(IMAGE_PATH)

In [56]:
image.shape
h, w = image.shape[:2]
print(f"{h} {w}")


187 270


In [57]:
#Function to preprocess the image(letterbox)


def letterbox_image(image, target_size=(224, 224), padding_color=(114, 114, 114)):
    pad_left  = 0
    pad_top  = 0
    SCALE = 1
    # 1. Calculate scaling ratio
    h, w = image.shape[:2]
    target_h, target_w = target_size
    ratio = min(target_w / w, target_h / h)
    SCALE=ratio
    print(f"Scale used: {ratio}")
    new_w, new_h = int(w * ratio), int(h * ratio)

    print(f"New Width: {new_w} New Height: {new_h} ")

    if target_h>new_h:
        print(f"Padding added to either height of image: {(target_h-new_h)//2}")
        pad_top = (target_h-new_h)/2
    if target_w>new_w:
        print(f"Padding added to either width of image: {(target_w-new_w)/2} ")
        pad_left = (target_w-new_w)/2
    # 2. Resize image
    resized_image = cv2.resize(image, (new_w, new_h))

    # 3. Create canvas
    canvas = np.full((target_h, target_w, 3), padding_color, dtype=np.uint8)

    # 4. Center image
    top = (target_h - new_h) // 2
    left = (target_w - new_w) // 2
    canvas[top:top+new_h, left:left+new_w] = resized_image

    return [canvas, pad_left, pad_top, SCALE]

In [58]:
resized_image = letterbox_image(image)[0]
resized_image.shape

Scale used: 0.8296296296296296
New Width: 224 New Height: 155 
Padding added to either height of image: 34


(224, 224, 3)

In [59]:
# Convert to RGB (OpenCV uses BGR by default)
rgb_image = cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB)

# Convert to float and Normalize to [0, 1]
# Important: YOLO models expect 0-1 range, not 0-255
float_image = rgb_image.astype(np.float32) / 255.0

In [60]:
#Flip the array and add a batch size at the beginning
flipped_image_array = np.transpose(float_image, (2,0,1))
formatted_image = flipped_image_array[np.newaxis, :]


In [61]:
#Create an inference session
session = ort.InferenceSession(MODEL_PATH, providers=['CPUExecutionProvider'])


In [62]:
#Run inference
input_name = session.get_inputs()[0].name
    
    # 3. Run Inference
print("\nRunning inference...")
outputs = session.run(None, {input_name: formatted_image})
raw_tensor = outputs[0] 


Running inference...


In [63]:
#Remove the first dimension and flip the outputs array
squeezed_array = np.squeeze(raw_tensor)
image_outputs = np.transpose(squeezed_array)
image_outputs.shape

(1029, 68)

In [64]:
#Find the rows with a confidence score above 0.5
column_index = 4
specific_criteria = 0.5

mask = image_outputs[:, column_index] >= specific_criteria

high_confidence_cells = image_outputs[mask]

high_confidence_cells.shape

(21, 68)

In [65]:
boxes = []
confidences = []
keypoints_list = []

for row in high_confidence_cells:
    # 1. Extract Bounding Box (Indices 0, 1, 2, 3)
    # The model gives [center_x, center_y, width, height]
    cx, cy, w, h = row[0:4]
    
    # 2. Convert to Top-Left format for OpenCV NMS
    # OpenCV expects [x, y, w, h] where (x,y) is the top-left corner
    x_min = cx - (w / 2)
    y_min = cy - (h / 2)
    
    # 3. Append to our lists
    # Note: We cast to float/int as required by OpenCV functions later
    boxes.append([float(x_min), float(y_min), float(w), float(h)])
    
    # 4. Extract Confidence (Index 4)
    confidences.append(float(row[4]))
    
    # 5. Extract Raw Keypoints (Indices 5 to 67)
    # We keep these as a flat array for now; we'll reshape them after NMS
    keypoints_list.append(row[5:68])

In [66]:
# Assuming 'boxes' and 'confidences' are the lists you populated in Phase 2
# These thresholds define how strict the NMS should be
score_threshold = 0.5  # Filter out boxes with low confidence
nms_threshold = 0.45   # If two boxes overlap by more than 45%, keep only the one with higher confidence

# 6. Execute NMS
# This returns the indices of the boxes that the algorithm decided to keep
indices = cv2.dnn.NMSBoxes(boxes, confidences, score_threshold, nms_threshold)

# 7. Filter by Index to find the "Winners"
final_results = []

if len(indices) > 0:
    # Flatten indices in case they are returned as a nested list [[0], [1]]
    for i in indices.flatten():
        # Retrieve the winning data from your original lists using the index
        winning_box = boxes[i]
        winning_score = confidences[i]
        winning_keypoints_flat = keypoints_list[i]
        
        # Store these together for the final phase
        final_results.append({
            "box": winning_box,
            "score": winning_score,
            "keypoints": winning_keypoints_flat
        })

print(f"Kept {len(final_results)} unique hand detections after NMS.")

Kept 4 unique hand detections after NMS.


In [67]:
final_results[0]

{'box': [131.5943603515625,
  73.60176086425781,
  43.29158020019531,
  57.54756164550781],
 'score': 0.7647603750228882,
 'keypoints': array([157.55298   , 123.85243   ,   0.95791554, 149.79962   ,
        120.63721   ,   0.9831357 , 145.69655   , 113.72596   ,
          0.989868  , 143.2107    , 102.407616  ,   0.97789073,
        144.51474   , 107.345146  ,   0.97995156, 152.43628   ,
         92.86058   ,   0.99648476, 150.80812   ,  87.16079   ,
          0.9951735 , 149.60213   ,  82.490425  ,   0.9882653 ,
        160.25189   , 104.562584  ,   0.9872331 , 159.23671   ,
         94.909744  ,   0.99715847, 148.01648   , 102.95915   ,
          0.9964098 , 145.19966   ,  93.79901   ,   0.99442244,
        143.45099   ,  88.70622   ,   0.98738754, 141.74886   ,
         84.20768   ,   0.98481274, 153.62347   , 102.70573   ,
          0.997284  , 157.62868   ,  89.697334  ,   0.99502695,
        156.69304   ,  84.97273   ,   0.98835504, 165.24481   ,
        107.58815   ,   0.9940260

In [68]:
# bounding_box_x = final_results[0]['box'][0]
# bounding_box_y = final_results[0]['box'][1]
# bounding_box_width
# box_centre = (bounding_box_x, bounding_box_y)

bounding_box = final_results[0]['box']
top_left = bounding_box[0]-bounding_box[2],bounding_box[1]-bounding_box[3]
top_left
bottom_right = bounding_box[0]+bounding_box[2],bounding_box[1]-bounding_box[3]
bottom_right
bounding_box

[131.5943603515625, 73.60176086425781, 43.29158020019531, 57.54756164550781]

In [69]:
# 1. Retrieve the metadata by unpacking the function return correctly
# (We don't need the image again since we have it, but we need the other values)
_, pad_left, pad_top, scale = letterbox_image(image)

print(f"Scale: {scale}, Pad Left: {pad_left}, Pad Top: {pad_top}")

# 2. Setup variables
winning_box = bounding_box
winning_keypoints_flat = final_results[0]['keypoints']

# 8. Reshape the Keypoints (Phase 4)
keypoints_2d = np.array(winning_keypoints_flat).reshape(21, 3)

# 10 & 11. Undo Padding and Scale for the Bounding Box
orig_x = (winning_box[0] - pad_left) / scale
orig_y = (winning_box[1] - pad_top) / scale
orig_w = winning_box[2] / scale
orig_h = winning_box[3] / scale

final_box = [int(orig_x), int(orig_y), int(orig_w), int(orig_h)]

# 10 & 11. Undo Padding and Scale for the Keypoints
keypoints_2d[:, 0] = (keypoints_2d[:, 0] - pad_left) / scale
keypoints_2d[:, 1] = (keypoints_2d[:, 1] - pad_top) / scale

final_keypoints = keypoints_2d.astype(int)

print(f"Original HD Box: {final_box}")
print(f"First HD Joint (X, Y, Vis): {final_keypoints[0]}")

Scale used: 0.8296296296296296
New Width: 224 New Height: 155 
Padding added to either height of image: 34
Scale: 0.8296296296296296, Pad Left: 0, Pad Top: 34.5
Original HD Box: [158, 47, 52, 69]
First HD Joint (X, Y, Vis): [189 107   0]


In [70]:
# (orig_x, orig_y)
# (o)

In [71]:
#Draw the rectangle on the original image.

original_image = cv2.imread(IMAGE_PATH)

# Extract the calculated integer coordinates from final_box
# final_box format is [x_min, y_min, width, height]
x, y, w, h = final_box

# Define points for cv2.rectangle (Top-Left and Bottom-Right)
pt1 = (x, y)
pt2 = (x + w, y + h)

# Draw the rectangle
cv2.rectangle(original_image, pt1, pt2, (0, 255, 0), 2)

# Draw keypoints
for i in range(len(final_keypoints)):
    kx, ky, vis = final_keypoints[i]
    cv2.circle(original_image, (kx, ky), 3, (0, 0, 255), -1)

cv2.imshow('Image', original_image)
k = cv2.waitKey(0)
cv2.destroyAllWindows()